# TrafficIQ — Sayan v1 (XGBoost)

**Improvements over baseline:**
- Smart missing-value flags before filling
- Context-aware fills (geohash → RoadType / Temperature / Weather)
- Geohash decoded to lat/lon coordinates
- OneHot encoding for RoadType
- Location-level and time-level demand statistics (target encoding)
- XGBoost with early stopping instead of Random Forest

**Metric:** `score = max(0, 100 × R²)`

## 0. Setup

In [1]:
# python-geohash ships as import geohash (not geohash2)
!pip install python-geohash -q


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geohash
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

ModuleNotFoundError: No module named 'xgboost'

## 1. Load Data

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head()

In [ ]:
# Missing values BEFORE cleaning
print('=== Missing values in TRAIN (before cleaning) ===')
print(train.isnull().sum())
print()
print('=== Missing values in TEST (before cleaning) ===')
print(test.isnull().sum())

## 2. Add Missing-Value Flags

Flag which rows had missing values **before** filling — this is a signal the model can use.
Must be done before any imputation.

In [ ]:
def add_missing_flags(df):
    """Add binary columns indicating where values were originally missing."""
    df['temp_was_missing']     = df['Temperature'].isna().astype(int)
    df['weather_was_missing']  = df['Weather'].isna().astype(int)
    df['roadtype_was_missing'] = df['RoadType'].isna().astype(int)
    return df

train = add_missing_flags(train)
test  = add_missing_flags(test)

print('Missing flags added.')
print(train[['temp_was_missing', 'weather_was_missing', 'roadtype_was_missing']].sum())

## 3. Smart Data Cleaning

**Rule:** All fill mappings are learned from TRAIN only, then applied to both train and test.

In [ ]:
# ── Learn all fill mappings from TRAIN only ──────────────────────────────────

# RoadType: same geohash → same road type
road_by_geo = (
    train.groupby('geohash')['RoadType']
    .agg(lambda x: x.mode()[0] if x.notna().any() else np.nan)
)
road_global_mode = train['RoadType'].mode()[0]

# Weather: same geohash → same typical weather
weather_by_geo = (
    train.groupby('geohash')['Weather']
    .agg(lambda x: x.mode()[0] if x.notna().any() else np.nan)
)
weather_global_mode = train['Weather'].mode()[0]

# Temperature: geohash median first, then weather-group median as fallback
temp_by_geo     = train.groupby('geohash')['Temperature'].median()
temp_by_weather = train.groupby('Weather')['Temperature'].median()
temp_global_med = train['Temperature'].median()

print('Fill mappings computed from train.')
print(f'  road_global_mode:    {road_global_mode}')
print(f'  weather_global_mode: {weather_global_mode}')
print(f'  temp_global_median:  {temp_global_med:.2f}')

In [ ]:
def apply_cleaning(df):
    """Apply pre-computed fill mappings (learned from train) to any dataframe."""
    df = df.copy()

    # RoadType: geohash lookup → global fallback
    df['RoadType'] = df['RoadType'].fillna(df['geohash'].map(road_by_geo))
    df['RoadType'] = df['RoadType'].fillna(road_global_mode)

    # Weather: geohash lookup → global fallback
    df['Weather'] = df['Weather'].fillna(df['geohash'].map(weather_by_geo))
    df['Weather'] = df['Weather'].fillna(weather_global_mode)

    # Temperature: geohash median → weather-group median → global median
    df['Temperature'] = df['Temperature'].fillna(df['geohash'].map(temp_by_geo))
    df['Temperature'] = df['Temperature'].fillna(df['Weather'].map(temp_by_weather))
    df['Temperature'] = df['Temperature'].fillna(temp_global_med)

    # LargeVehicles and Landmarks — fill with mode (rarely missing)
    for col in ['LargeVehicles', 'Landmarks']:
        if col in df.columns:
            df[col] = df[col].fillna(train[col].mode()[0])

    return df

train = apply_cleaning(train)
test  = apply_cleaning(test)

print('=== Missing values AFTER cleaning ===')
cols_to_check = ['RoadType', 'Weather', 'Temperature', 'LargeVehicles', 'Landmarks']
print(train[cols_to_check].isnull().sum())
print()
print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

## 4. Feature Engineering

### 4a. Time Features

In [ ]:
def add_time_features(df):
    # Parse 'H:MM' format into integer hour and minute
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour']      = parts[0]
    df['minute']    = parts[1]
    # 96 slots of 15 minutes each across a day
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15

    # Cyclical encoding captures the 23:45 → 0:00 wrap-around
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # Day of week proxy (assumes day numbering is consistent)
    df['day_of_week'] = df['day'] % 7

    return df

train = add_time_features(train)
test  = add_time_features(test)

print('Time features added.')
print(train[['hour', 'minute', 'time_slot', 'hour_sin', 'hour_cos', 'day_of_week']].head())

### 4b. Binary Flags

In [ ]:
def add_binary_flags(df):
    df['is_large_vehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['has_landmark']      = (df['Landmarks'] == 'Yes').astype(int)
    return df

train = add_binary_flags(train)
test  = add_binary_flags(test)
print('Binary flags added.')

### 4c. Geohash → Lat/Lon

In [ ]:
def decode_geohash(df):
    # Decode each geohash string into geographic coordinates
    coords = df['geohash'].apply(lambda h: geohash.decode(h))
    df['lat'] = coords.apply(lambda c: c[0])
    df['lon'] = coords.apply(lambda c: c[1])
    return df

train = decode_geohash(train)
test  = decode_geohash(test)

print('Geohash decoded to lat/lon.')
print(train[['geohash', 'lat', 'lon']].head())

### 4d. RoadType OneHot Encoding

OneHot is better than LabelEncoder for nominal categoricals — it avoids implying an ordering.

In [ ]:
# Compute dummies on train to establish the canonical column set
road_dummies_train = pd.get_dummies(train['RoadType'], prefix='road')
road_dummies_test  = pd.get_dummies(test['RoadType'],  prefix='road')

# Align test to train columns — fills 0 for any category not seen in test
road_dummies_test = road_dummies_test.reindex(columns=road_dummies_train.columns, fill_value=0)

train = pd.concat([train, road_dummies_train], axis=1)
test  = pd.concat([test,  road_dummies_test],  axis=1)

road_dummy_cols = list(road_dummies_train.columns)
print(f'RoadType dummy columns: {road_dummy_cols}')

### 4e. Weather Label Encoding

In [ ]:
# Fit label encoder on train categories only
le_weather = LabelEncoder()
le_weather.fit(train['Weather'])

train['Weather_enc'] = le_weather.transform(train['Weather'])
# Map unseen test values to 0 (rare edge case)
test['Weather_enc'] = test['Weather'].apply(
    lambda x: le_weather.transform([x])[0] if x in le_weather.classes_ else 0
)

print('Weather encoding:', dict(zip(le_weather.classes_, le_weather.transform(le_weather.classes_))))

## 5. Target Encoding — Demand Statistics

**All stats computed from TRAIN only**, then mapped to both train and test.

In [ ]:
# ── Location-level demand stats (per geohash) ─────────────────────────────────
geo_stats = (
    train.groupby('geohash')['demand']
    .agg(['mean', 'median', 'std'])
    .rename(columns={'mean': 'geo_mean_demand',
                     'median': 'geo_median_demand',
                     'std':    'geo_std_demand'})
    .reset_index()
)

train = train.merge(geo_stats, on='geohash', how='left')
test  = test.merge(geo_stats,  on='geohash', how='left')

# Fill any geohashes in test not seen in train with global stats
for col in ['geo_mean_demand', 'geo_median_demand', 'geo_std_demand']:
    global_val = train[col].mean()
    train[col] = train[col].fillna(global_val)
    test[col]  = test[col].fillna(global_val)

print('Geo demand stats added.')
print(train[['geohash', 'geo_mean_demand', 'geo_median_demand', 'geo_std_demand']].head())

In [ ]:
# ── Time-slot level demand stats ──────────────────────────────────────────────
time_stats = (
    train.groupby('time_slot')['demand']
    .mean()
    .rename('time_mean_demand')
)

train['time_mean_demand'] = train['time_slot'].map(time_stats)
test['time_mean_demand']  = test['time_slot'].map(time_stats)

# Fallback for any unseen slots
global_time_mean = time_stats.mean()
train['time_mean_demand'] = train['time_mean_demand'].fillna(global_time_mean)
test['time_mean_demand']  = test['time_mean_demand'].fillna(global_time_mean)

print('Time-slot demand stats added.')

In [ ]:
# ── RoadType × time_slot combo demand stats ───────────────────────────────────
road_time_stats = (
    train.groupby(['RoadType', 'time_slot'])['demand']
    .mean()
    .rename('road_time_demand')
    .reset_index()
)

train = train.merge(road_time_stats, on=['RoadType', 'time_slot'], how='left')
test  = test.merge(road_time_stats,  on=['RoadType', 'time_slot'], how='left')

# Fallback for unseen combos
global_rt_mean = train['road_time_demand'].mean()
train['road_time_demand'] = train['road_time_demand'].fillna(global_rt_mean)
test['road_time_demand']  = test['road_time_demand'].fillna(global_rt_mean)

print('RoadType×time_slot demand stats added.')
print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

## 6. Build Feature Matrix

In [ ]:
FEATURES = [
    # Time
    'day', 'day_of_week', 'hour', 'minute', 'time_slot', 'hour_sin', 'hour_cos',
    # Location
    'lat', 'lon',
    # Road
    'NumberofLanes', 'is_large_vehicles', 'has_landmark',
    # Weather
    'Temperature', 'Weather_enc',
    # Missing flags
    'temp_was_missing', 'weather_was_missing', 'roadtype_was_missing',
    # Target encoding
    'geo_mean_demand', 'geo_median_demand', 'geo_std_demand',
    'time_mean_demand', 'road_time_demand',
] + road_dummy_cols  # OneHot road type columns

TARGET = 'demand'

X      = train[FEATURES]
y      = train[TARGET]
X_test = test[FEATURES]

print(f'Features: {len(FEATURES)}')
print(f'X shape:      {X.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'\nFeature list: {FEATURES}')

In [ ]:
# 80/20 train-val split (stratified split not needed for regression)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')

## 7. Train — XGBoost with Early Stopping

In [ ]:
model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,           # row sampling per tree
    colsample_bytree=0.8,    # feature sampling per tree
    min_child_weight=5,      # minimum sum of instance weight in a child
    gamma=0.1,               # minimum loss reduction for a split
    reg_alpha=0.1,           # L1 regularisation
    reg_lambda=1.0,          # L2 regularisation
    early_stopping_rounds=50,
    random_state=SEED,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

print(f'\nBest iteration: {model.best_iteration}')

## 8. Validate Locally

In [ ]:
val_preds = model.predict(X_val)
r2    = r2_score(y_val, val_preds)
score = max(0.0, 100 * r2)

print('=' * 40)
print(f'  R²:              {r2:.4f}')
print(f'  Competition score: {score:.2f}')
print('=' * 40)
print()
print('⚠️  Only submit if this score beats our current best.')

In [ ]:
# Feature importance
importances = (
    pd.Series(model.feature_importances_, index=FEATURES)
    .sort_values(ascending=True)
)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('XGBoost Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter
plt.figure(figsize=(5, 5))
plt.scatter(y_val, val_preds, alpha=0.15, s=4, color='steelblue')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1)
plt.xlabel('Actual demand')
plt.ylabel('Predicted demand')
plt.title(f'Actual vs Predicted  (Score: {score:.1f})')
plt.tight_layout()
plt.show()

## 9. Generate Submission

In [ ]:
test_preds = model.predict(X_test)

# Clip to valid demand range
test_preds = np.clip(test_preds, 0.0, 1.0)

submission = pd.DataFrame({
    'Index':  test['Index'],
    'demand': test_preds,
})

print(f'Submission shape: {submission.shape}  (expected: 41778 × 2)')
submission.head()

In [ ]:
submission_path = f'../submissions/sayan1_xgboost_score{score:.1f}.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved: {submission_path}')